In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-469mblvt/unsloth_8e31874f8dbc45bcab4db7b5c6450035
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-469mblvt/unsloth_8e31874f8dbc45bcab4db7b5c6450035
  Resolved https://github.com/unslothai/unsloth.git to commit abeabc71bb3bb85e0d006f00a9c665842ddb820c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.8 MB/s eta 0:00:0

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # Reducir para evitar problemas de memoria
dtype = None
load_in_4bit = True

print("Loading base Llama 3 (8B) quantized model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = "auto",   # Ayuda evitar fragmentación
)

FastLanguageModel.for_inference(model)
print("Model loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base Llama 3 (8B) quantized model...
==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded successfully on GPU!


In [ ]:
import pandas as pd
import time
from copy import deepcopy
from google.colab import drive
import os

# 1. Definimos los 10 pacientes (en inglés para mantener parámetros iguales en el modelo con finetunning)
patients_base = [
    {
        "id_base": "Case_001_Lupus",
        "specialty": "Rheumatology",
        "real_disease": "Systemic Lupus Erythematosus (SLE)",
        "symptoms": "28-year-old woman. Presents with migratory joint pain, extreme fatigue of months of evolution, painless oral ulcers, and a reddish rash on the cheeks that worsens with sun exposure."
    },
    {
        "id_base": "Case_002_Pheochromocytoma",
        "specialty": "Endocrinology",
        "real_disease": "Pheochromocytoma",
        "symptoms": "45-year-old man. Presents with sudden episodes of throbbing headache, profuse sweating, and severe palpitations lasting 15 minutes. During the episode in the emergency room, his blood pressure is 190/110 mmHg."
    },
    {
        "id_base": "Case_003_Endocarditis",
        "specialty": "Infectious Diseases/Cardiology",
        "real_disease": "Infective Endocarditis",
        "symptoms": "35-year-old man, intravenous drug user. Presents with intermittent fever for 3 weeks, appearance of a new heart murmur, and small reddish lesions on the palms (Janeway lesions)."
    },
    {
        "id_base": "Case_004_Myasthenia",
        "specialty": "Neurology",
        "real_disease": "Myasthenia Gravis",
        "symptoms": "32-year-old woman. Reports double vision (diplopia) and drooping eyelids (ptosis) that worsen notably at the end of the day or after reading for a while. Recently has difficulty swallowing solids."
    },
    {
        "id_base": "Case_005_Embolism",
        "specialty": "Pulmonology",
        "real_disease": "Pulmonary Thromboembolism (PTE)",
        "symptoms": "60-year-old woman, hip surgery 10 days ago. Presents with sudden onset dyspnea, chest pain that worsens with deep inspiration (pleuritic), and tachycardia at 120 bpm. No fever."
    },
    {
        "id_base": "Case_006_Lyme",
        "specialty": "Infectious Diseases",
        "real_disease": "Lyme Disease",
        "symptoms": "40-year-old man, frequent camper. Presents with fever, myalgias, and an expanding target-shaped skin lesion (erythema migrans) on the thigh, without significant pain or itching."
    },
    {
        "id_base": "Case_007_Sclerosis",
        "specialty": "Neurology",
        "real_disease": "Multiple Sclerosis",
        "symptoms": "25-year-old woman. Painful vision loss in the left eye (optic neuritis) established in 2 days. Reports that a year ago she had an episode of tingling in both legs that disappeared on its own."
    },
    {
        "id_base": "Case_008_Addison",
        "specialty": "Endocrinology",
        "real_disease": "Addison's Disease",
        "symptoms": "50-year-old man. Presents with extreme fatigue, weight loss, unusual craving for salty foods, and hyperpigmentation (darkening) in the hand folds and gums. Low blood pressure (85/50)."
    },
    {
        "id_base": "Case_009_Ectopic",
        "specialty": "Gynecology",
        "real_disease": "Ruptured Ectopic Pregnancy",
        "symptoms": "29-year-old woman. Missed period for 6 weeks. Presents with acute, sharp abdominal pain in the right iliac fossa, intense dizziness, and pain radiating to the right shoulder."
    },
    {
        "id_base": "Case_010_GuillainBarre",
        "specialty": "Neurology",
        "real_disease": "Guillain-Barré Syndrome",
        "symptoms": "38-year-old man. Two weeks ago had gastroenteritis. Now presents with progressive muscle weakness that started in the feet and is ascending toward the thighs, with absence of patellar reflexes."
    }
]

# 10 intentos por pacientes para un total de 100 intentos
patients_test = []
for attempt in range(1, 11):
    for patient in patients_base:
        clone = deepcopy(patient)
        clone["id"] = f"{clone['id_base']}_Attempt_{attempt:02d}"
        patients_test.append(clone)

patients_test = sorted(patients_test, key=lambda x: x['id'])
evaluation_results = []

print(f"Starting evaluation with BASE model: {len(patients_test)} total inferences...\n")

for i, patient in enumerate(patients_test):
    print(f"[{i+1}/100] Processing {patient['id']}...")

    prompt_dynamic = f"""You are an expert physician. Analyze this complex clinical case rigorously.

Clinical Case: {patient['symptoms']}

Provide exclusively:
1. Main diagnostic suspicion.
2. Brief medical justification.

Response:"""

    inputs = tokenizer([prompt_dynamic], return_tensors="pt").to("cuda")
    start_time = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    elapsed_time = time.time() - start_time
    full_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    clean_response = full_response[len(prompt_dynamic):].strip()

    evaluation_results.append({
        "ID_Case": patient["id"],
        "Specialty": patient["specialty"],
        "Real_Diagnosis": patient["real_disease"],
        "Response_BASE_Model": clean_response,
        "Inference_Time_sec": round(elapsed_time, 2)
    })

# Guardar en Drive
drive.mount('/content/drive')
df_results = pd.DataFrame(evaluation_results)

output_dir = '/content/drive/MyDrive/Bioinformática_LLM'
os.makedirs(output_dir, exist_ok=True)

save_path = f'{output_dir}/resultados_modelo_base_complejos.csv'
df_results.to_csv(save_path, index=False, encoding='utf-8')

print("\n" + "="*50)
print(f"Saved to: {save_path}")
print(f"\nFirst 5 results preview:")
display(df_results.head())

Starting rigorous evaluation with BASE model: 100 total inferences...

This will take about 15-25 minutes. Do not close the tab.

[1/100] Processing Case_001_Lupus_Attempt_01...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

[2/100] Processing Case_001_Lupus_Attempt_02...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3/100] Processing Case_001_Lupus_Attempt_03...


...

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[99/100] Processing Case_010_GuillainBarre_Attempt_09...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[100/100] Processing Case_010_GuillainBarre_Attempt_10...
Mounted at /content/drive

Evaluation completed! Saved to: /content/drive/MyDrive/Bioinformática_LLM/resultados_modelo_base_complejos.csv

First 5 results preview:


,ID_Case,Specialty,Real_Diagnosis,Response_BASE_Model,Inference_Time_sec
0,Case_001_Lupus_Attempt_01,Rheumatology,Systemic Lupus Erythematosus (SLE),1. Main diagnostic suspicion: Sjögren's syndro...,16.75
1,Case_001_Lupus_Attempt_02,Rheumatology,Systemic Lupus Erythematosus (SLE),1. Main diagnostic suspicion: Juvenile Idiopat...,8.71
2,Case_001_Lupus_Attempt_03,Rheumatology,Systemic Lupus Erythematosus (SLE),**Main diagnostic suspicion:** Systemic Lupus ...,8.71
3,Case_001_Lupus_Attempt_04,Rheumatology,Systemic Lupus Erythematosus (SLE),1. Main diagnostic suspicion: Adult-onset Stil...,8.64
4,Case_001_Lupus_Attempt_05,Rheumatology,Systemic Lupus Erythematosus (SLE),1. Main diagnostic suspicion: Autoimmune conne...,9.01
